# 04 — Entrenamiento CTGAN y TVAE

**Fase 2 — Modelos generativos tabulares (baseline)**  
Objetivo: entrenar CTGAN y TVAE sobre `tabular_48h.parquet`, generar muestras sintéticas
incondicionales y condicionales por mortalidad, y hacer una validación rápida de distribuciones
antes de la evaluación formal de la Fase 3.

Pasos:
1. Clasificación de columnas y definición de metadatos SDV
2. Entrenamiento CTGAN con hiperparámetros justificados
3. Entrenamiento TVAE con hiperparámetros justificados
4. Generación incondicional y condicional por mortalidad
5. Validación rápida de distribuciones marginales
6. Guardado de modelos y datasets sintéticos

## 0. Imports y configuración

In [ ]:
import sys, subprocess, warnings, time
warnings.filterwarnings("ignore")

# Instalar SDV si no está disponible (primera ejecución en servidor)
try:
    import sdv
    print(f"SDV {sdv.__version__} ya instalado.")
except ImportError:
    print("Instalando SDV...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sdv", "-q"])
    import sdv
    print(f"SDV {sdv.__version__} instalado.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
from sdv.sampling import Condition

ROOT      = Path("..")
PROCESSED = ROOT / "data" / "processed"
SYNTHETIC = ROOT / "data" / "synthetic"
MODELS    = ROOT / "models"
REPORTS   = ROOT / "reports"

for d in [SYNTHETIC, MODELS, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")
print("Entorno listo.")

## 1. Carga y clasificación de columnas

In [ ]:
tab = pd.read_parquet(PROCESSED / "tabular_48h.parquet")
print(f"Shape: {tab.shape}")
print(f"Dtypes: {tab.dtypes.value_counts().to_dict()}")

TARGET = "hospital_expire_flag"

# Columnas de identificación — excluir de la síntesis
id_cols = [c for c in tab.columns if c.endswith("_id")]

# Binarias: exactamente 2 valores únicos (incluye target y flags de diagnóstico)
binary_cols = [
    c for c in tab.columns
    if c not in id_cols and tab[c].dropna().nunique() <= 2
]

# Categóricas de texto (género, etnia, tipo de admisión, etc.)
cat_cols = [
    c for c in tab.columns
    if c not in id_cols + binary_cols and tab[c].dtype == object
]

# Numéricas continuas: el resto
num_cols = [
    c for c in tab.columns
    if c not in id_cols + binary_cols + cat_cols
]

print(f"\nColumnas ID:           {len(id_cols)}  → {id_cols}")
print(f"Columnas binarias:     {len(binary_cols)}")
print(f"Columnas categóricas:  {len(cat_cols)}  → {cat_cols}")
print(f"Columnas numéricas:    {len(num_cols)}")
print(f"Total:                 {len(id_cols)+len(binary_cols)+len(cat_cols)+len(num_cols)} / {tab.shape[1]}")

## 2. Definición de metadatos SDV

SDV usa un objeto `SingleTableMetadata` para conocer el tipo semántico de cada columna.
La detección automática es el punto de partida; los overrides manuales corrigen los tipos
que SDV infiere mal (binarias 0/1 detectadas como numéricas, IDs detectados como numéricos).

In [ ]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(tab)

# IDs → excluir de síntesis
for col in id_cols:
    metadata.update_column(col, sdtype="id")
if "icustay_id" in tab.columns:
    metadata.set_primary_key("icustay_id")

# Binarias 0/1 → categorical (SDV modela mejor la distribución discreta)
for col in binary_cols:
    metadata.update_column(col, sdtype="categorical")

# Categóricas de texto → categorical
for col in cat_cols:
    metadata.update_column(col, sdtype="categorical")

# Numéricas → numerical
for col in num_cols:
    metadata.update_column(col, sdtype="numerical")

# Resumen
sdtype_counts = {}
for col, info in metadata.columns.items():
    st = info.get("sdtype", "unknown")
    sdtype_counts[st] = sdtype_counts.get(st, 0) + 1

print("Metadatos SDV definidos.")
print(f"Primary key: {metadata.primary_key}")
print("\nResumen de sdtypes:")
for k, v in sorted(sdtype_counts.items()):
    print(f"  {k:<15} {v} columnas")

# Validar que la metadata es coherente
metadata.validate()
print("\nValidación de metadata: OK")

## 3. Entrenamiento CTGAN

### Justificación de hiperparámetros

| Parámetro | Valor | Justificación |
|---|---|---|
| `embedding_dim` | 128 | Dimensión del vector de ruido; 128 captura la variabilidad de 127 features sin sobreajuste |
| `generator_dim` | (256, 256) | 2 capas × 256 neuronas; capacidad suficiente para el espacio clínico mixto |
| `discriminator_dim` | (256, 256) | Simétrico al generador para equilibrio GAN estable |
| `batch_size` | 500 | ~2.2% del dataset; gradientes estables con buen throughput |
| `epochs` | 500 | Por encima del default (300) para asegurar convergencia con 22k muestras |
| `pac` | 10 | Packing: el discriminador evalúa grupos de 10, reduce mode collapse en variables correladas |
| `cuda` | True | GPU si disponible (servidor universitario) |

In [ ]:
CTGAN_PARAMS = dict(
    embedding_dim=128,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    batch_size=500,
    epochs=500,
    pac=10,
    verbose=True,
    cuda=True,
)

print("Parámetros CTGAN:")
for k, v in CTGAN_PARAMS.items():
    print(f"  {k:<25} {v}")

ctgan = CTGANSynthesizer(metadata, **CTGAN_PARAMS)

print("\nEntrenando CTGAN...")
t0 = time.time()
ctgan.fit(tab)
elapsed_ctgan = time.time() - t0
print(f"\nEntrenamiento completado en {elapsed_ctgan/60:.1f} min.")

ctgan.save(str(MODELS / "ctgan.pkl"))
print("Modelo guardado: models/ctgan.pkl")

In [ ]:
# Curvas de pérdida — diagnóstico de convergencia
try:
    loss_df = ctgan.get_loss_values()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(loss_df["Epoch"], loss_df["Generator Loss"],
            label="Generator", color="steelblue", linewidth=1.2)
    ax.plot(loss_df["Epoch"], loss_df["Discriminator Loss"],
            label="Discriminator", color="tomato", linewidth=1.2)
    ax.set_xlabel("Época")
    ax.set_ylabel("Loss")
    ax.set_title("Curvas de pérdida — CTGAN")
    ax.legend()
    plt.tight_layout()
    plt.savefig(REPORTS / "ctgan_loss_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figura guardada: reports/ctgan_loss_curves.png")
except Exception as e:
    print(f"Loss curves no disponibles en esta versión de SDV: {e}")

## 4. Entrenamiento TVAE

### Justificación de hiperparámetros

| Parámetro | Valor | Justificación |
|---|---|---|
| `embedding_dim` | 128 | Igual que CTGAN para comparabilidad directa |
| `compress_dims` | (256, 256) | Encoder con 2 capas; mismo orden de magnitud que el generador CTGAN |
| `decompress_dims` | (256, 256) | Decoder simétrico al encoder |
| `l2scale` | 1e-5 | Regularización L2 leve para evitar sobreajuste sin penalizar la reconstrucción |
| `batch_size` | 500 | Igual que CTGAN para comparabilidad |
| `epochs` | 500 | Igual que CTGAN |
| `cuda` | True | GPU si disponible |

In [ ]:
TVAE_PARAMS = dict(
    embedding_dim=128,
    compress_dims=(256, 256),
    decompress_dims=(256, 256),
    l2scale=1e-5,
    batch_size=500,
    epochs=500,
    verbose=True,
    cuda=True,
)

print("Parámetros TVAE:")
for k, v in TVAE_PARAMS.items():
    print(f"  {k:<25} {v}")

tvae = TVAESynthesizer(metadata, **TVAE_PARAMS)

print("\nEntrenando TVAE...")
t0 = time.time()
tvae.fit(tab)
elapsed_tvae = time.time() - t0
print(f"\nEntrenamiento completado en {elapsed_tvae/60:.1f} min.")

tvae.save(str(MODELS / "tvae.pkl"))
print("Modelo guardado: models/tvae.pkl")

In [ ]:
try:
    loss_df_tvae = tvae.get_loss_values()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(loss_df_tvae["Epoch"], loss_df_tvae["Loss"],
            label="ELBO Loss", color="seagreen", linewidth=1.2)
    ax.set_xlabel("Época")
    ax.set_ylabel("Loss")
    ax.set_title("Curva de pérdida — TVAE (ELBO)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(REPORTS / "tvae_loss_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figura guardada: reports/tvae_loss_curve.png")
except Exception as e:
    print(f"Loss curve no disponible en esta versión de SDV: {e}")

## 5. Generación de muestras sintéticas

Se generan dos tipos de muestras para cada modelo:

- **Incondicional**: N = tamaño del dataset real, distribución libre.
- **Condicional por mortalidad**: se fija `hospital_expire_flag` al valor real observado
  (n_fallecidos y n_supervivientes). Permite evaluar si el modelo reproduce correctamente
  los patrones clínicos de cada subgrupo.

In [ ]:
N = len(tab)

print(f"Generando {N:,} muestras incondicionales con CTGAN...")
ctgan_samples = ctgan.sample(num_rows=N)
ctgan_samples.to_parquet(SYNTHETIC / "ctgan_samples.parquet", index=False)
print(f"  Guardado: data/synthetic/ctgan_samples.parquet  {ctgan_samples.shape}")

print(f"\nGenerando {N:,} muestras incondicionales con TVAE...")
tvae_samples = tvae.sample(num_rows=N)
tvae_samples.to_parquet(SYNTHETIC / "tvae_samples.parquet", index=False)
print(f"  Guardado: data/synthetic/tvae_samples.parquet  {tvae_samples.shape}")

In [ ]:
n_fallecidos     = int(tab[TARGET].sum())
n_supervivientes = len(tab) - n_fallecidos
print(f"Real — supervivientes: {n_supervivientes:,}  |  fallecidos: {n_fallecidos:,}  ({tab[TARGET].mean()*100:.1f}%)")

cond_vivos    = Condition(num_rows=n_supervivientes, column_values={TARGET: 0})
cond_fallecidos = Condition(num_rows=n_fallecidos,   column_values={TARGET: 1})

print("\nGenerando muestras condicionales con CTGAN...")
ctgan_cond = ctgan.sample_from_conditions(conditions=[cond_vivos, cond_fallecidos])
ctgan_cond.to_parquet(SYNTHETIC / "ctgan_samples_conditional.parquet", index=False)
print(f"  Shape: {ctgan_cond.shape}  |  Mortalidad: {ctgan_cond[TARGET].mean()*100:.1f}%")

print("\nGenerando muestras condicionales con TVAE...")
tvae_cond = tvae.sample_from_conditions(conditions=[cond_vivos, cond_fallecidos])
tvae_cond.to_parquet(SYNTHETIC / "tvae_samples_conditional.parquet", index=False)
print(f"  Shape: {tvae_cond.shape}  |  Mortalidad: {tvae_cond[TARGET].mean()*100:.1f}%")

## 6. Validación rápida de distribuciones

Comparación visual de distribuciones marginales y estadísticos descriptivos.
La evaluación cuantitativa formal (TVD, correlaciones, métricas SDMetrics) se realiza en la Fase 3.

In [ ]:
VITALS_PLOT = [
    "heart_rate_mean", "sbp_mean", "spo2_mean", "gcs_total_mean",
    "resp_rate_mean", "lactate_mean", "creatinine_mean", "age"
]
vitals_plot = [c for c in VITALS_PLOT if c in tab.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, vitals_plot):
    real_vals  = tab[col].dropna()
    ctgan_vals = ctgan_samples[col].dropna()
    tvae_vals  = tvae_samples[col].dropna()

    ax.hist(real_vals,  bins=60, alpha=0.5, density=True, color="steelblue", label="Real")
    ax.hist(ctgan_vals, bins=60, alpha=0.5, density=True, color="tomato",    label="CTGAN")
    ax.hist(tvae_vals,  bins=60, alpha=0.5, density=True, color="seagreen",  label="TVAE")
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7)
for j in range(len(vitals_plot), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribuciones marginales — Real vs CTGAN vs TVAE (incondicional)", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / "ctgan_tvae_marginals.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/ctgan_tvae_marginals.png")

In [ ]:
# Comparación de mortalidad (target)
print("Mortalidad (hospital_expire_flag):")
print(f"  Real:                 {tab[TARGET].mean():.4f}  ({tab[TARGET].mean()*100:.1f}%)")
print(f"  CTGAN incondicional:  {ctgan_samples[TARGET].mean():.4f}  ({ctgan_samples[TARGET].mean()*100:.1f}%)")
print(f"  TVAE incondicional:   {tvae_samples[TARGET].mean():.4f}  ({tvae_samples[TARGET].mean()*100:.1f}%)")
print(f"  CTGAN condicional:    {ctgan_cond[TARGET].mean():.4f}  ({ctgan_cond[TARGET].mean()*100:.1f}%)")
print(f"  TVAE condicional:     {tvae_cond[TARGET].mean():.4f}  ({tvae_cond[TARGET].mean()*100:.1f}%)")

# Estadísticos descriptivos de variables clave
key_cols = [c for c in ["heart_rate_mean", "sbp_mean", "lactate_mean",
                         "creatinine_mean", "age", "los"] if c in tab.columns]

rows = []
for col in key_cols:
    rows.append({
        "variable":    col,
        "real_mean":   tab[col].mean(),
        "real_std":    tab[col].std(),
        "ctgan_mean":  ctgan_samples[col].mean(),
        "ctgan_std":   ctgan_samples[col].std(),
        "tvae_mean":   tvae_samples[col].mean(),
        "tvae_std":    tvae_samples[col].std(),
    })

stats_df = pd.DataFrame(rows).set_index("variable").round(3)
print("\nEstadísticos descriptivos (media ± desv.):")
print(stats_df.to_string())
stats_df.to_csv(REPORTS / "ctgan_tvae_stats_comparison.csv")
print("\nGuardado: reports/ctgan_tvae_stats_comparison.csv")

## 7. Resumen

In [ ]:
print("=" * 60)
print("  RESUMEN — Notebook 04")
print("=" * 60)
checks = [
    ("Dataset real",              f"{tab.shape}"),
    ("Mortalidad real",           f"{tab[TARGET].mean()*100:.1f}%"),
    ("CTGAN — tiempo entren.",    f"{elapsed_ctgan/60:.1f} min"),
    ("TVAE  — tiempo entren.",    f"{elapsed_tvae/60:.1f} min"),
    ("CTGAN muestras incond.",    f"{ctgan_samples.shape}"),
    ("TVAE  muestras incond.",    f"{tvae_samples.shape}"),
    ("CTGAN muestras cond.",      f"{ctgan_cond.shape}"),
    ("TVAE  muestras cond.",      f"{tvae_cond.shape}"),
    ("Mortalidad CTGAN incond.",  f"{ctgan_samples[TARGET].mean()*100:.1f}%"),
    ("Mortalidad TVAE incond.",   f"{tvae_samples[TARGET].mean()*100:.1f}%"),
]
for name, val in checks:
    print(f"  {name:<32} {val}")
print("=" * 60)
print("Listos para notebook 05 (TabDDPM).")